In [1]:
import pandas as pd

In [5]:
dfrt= pd.read_csv("MTA_RIDE_TIME_2017.csv")
dfrt

/var/folders/z2/z2mrld5d7gx6zgg9qmsyx67w0000gn/T/ipykernel_57025/1448752207.py:1: DtypeWarning: Columns (0: Total Trips) have mixed types. Specify dtype option on import or set low_memory=False.
  dfrt= pd.read_csv("MTA_RIDE_TIME_2017.csv")


,Trip Date,Peak Hour,Provider Type,Origin,destination,Distance,Percent of Scheduled Ride Time Category,Within Max Ride Time,Total Trips,Total Ride Time (Min)
0,01/26/2017,Peak,Primary,Bronx,Queens,6-9 miles,<=100%,True,3,113
1,01/31/2017,Peak,Primary,Bronx,Bronx,9-12 miles,<=100%,True,1,84
2,02/09/2017,Peak,Primary,Bronx,Bronx,0-3 miles,<=100%,True,45,798
3,02/10/2017,Peak,Primary,Queens,Bronx,6-9 miles,<=100%,True,2,76
4,02/13/2017,Peak,Primary,Queens,Bronx,3-6 miles,<=100%,True,1,22
...,...,...,...,...,...,...,...,...,...,...
4304366,02/28/2026,Off-Peak,Primary,Bronx,Bronx,6-9 miles,125-150%,True,1,75
4304367,02/28/2026,Off-Peak,Primary,Nassau,Brooklyn,6-9 miles,125-150%,True,1,65
4304368,02/28/2026,Off-Peak,Primary,Staten Island,Brooklyn,6-9 miles,125-150%,True,1,44
4304369,02/28/2026,Off-Peak,Primary,Bronx,Westchester,6-9 miles,125-150%,True,1,59


In [6]:
dfrt.info()

<class 'pandas.DataFrame'>
RangeIndex: 4304371 entries, 0 to 4304370
Data columns (total 10 columns):
 #   Column                                   Dtype 
---  ------                                   ----- 
 0   Trip Date                                str   
 1   Peak Hour                                str   
 2   Provider Type                            str   
 3   Origin                                   str   
 4   destination                              str   
 5   Distance                                 str   
 6   Percent of Scheduled Ride Time Category  str   
 7   Within Max Ride Time                     bool  
 8   Total Trips                              object
 9   Total Ride Time (Min)                    str   
dtypes: bool(1), object(1), str(8)
memory usage: 299.7+ MB


In [8]:
# Load the data (using the quick fix to suppress the warning)
dfrt = pd.read_csv("MTA_RIDE_TIME_2017.csv", low_memory=False)

# Force the column to be numeric, turning bad text into NaN
dfrt['Total Trips'] = pd.to_numeric(dfrt['Total Trips'], errors='coerce')

In [10]:
dfrt.info()

<class 'pandas.DataFrame'>
RangeIndex: 4304371 entries, 0 to 4304370
Data columns (total 10 columns):
 #   Column                                   Dtype  
---  ------                                   -----  
 0   Trip Date                                str    
 1   Peak Hour                                str    
 2   Provider Type                            str    
 3   Origin                                   str    
 4   destination                              str    
 5   Distance                                 str    
 6   Percent of Scheduled Ride Time Category  str    
 7   Within Max Ride Time                     bool   
 8   Total Trips                              float64
 9   Total Ride Time (Min)                    str    
dtypes: bool(1), float64(1), str(8)
memory usage: 299.7 MB


In [13]:
##Export sample
dfrt_sample = dfrt.sample(300)
dfrt_sample.to_csv("dfrt_sample.csv", index = False)

In [14]:
dfrt["Percent of Scheduled Ride Time Category"].unique()

<StringArray>
['<=100%', '125-150%', 'Over 200%', '100-125%', '150-200%']
Length: 5, dtype: str

In [19]:
# 1. Load the dataset
dfrt = pd.read_csv("MTA_RIDE_TIME_2017.csv", low_memory=False)

# 2. Clean 'Total Trips' to be safely numeric
dfrt['Total Trips'] = pd.to_numeric(dfrt['Total Trips'], errors='coerce')

# 3. Convert 'Trip Date' to datetime format, then extract Year and Month
dfrt['Trip Date'] = pd.to_datetime(dfrt['Trip Date'], errors='coerce')
dfrt['Year'] = dfrt['Trip Date'].dt.year            # Extract the Year (e.g., 2017)
dfrt['Month_Num'] = dfrt['Trip Date'].dt.month      # Extract the Month Number for sorting (1-12)
dfrt['Month'] = dfrt['Trip Date'].dt.strftime('%b') # Extract the Month Name for display (Jan, Feb)

# 4. Filter for 'Broker' and 'Primary'
dfrt_filtered = dfrt[dfrt['Provider Type'].isin(['Broker', 'Primary'])]

# 5. Group by Year, Month, and Provider Type to get the total sum of all trips
total_trips = dfrt_filtered.groupby(['Year', 'Month_Num', 'Month', 'Provider Type'])['Total Trips'].sum()

# 6. Get the sum of trips ONLY for rows that are 'Over 200%'
over_200_mask = dfrt_filtered['Percent of Scheduled Ride Time Category'] == 'Over 200%'
over_200_trips = dfrt_filtered[over_200_mask].groupby(['Year', 'Month_Num', 'Month', 'Provider Type'])['Total Trips'].sum()

# 7. Calculate the percentage and fill any empty (NaN) values with 0
pct_over_200 = (over_200_trips / total_trips * 100).fillna(0).reset_index()
pct_over_200.rename(columns={'Total Trips': 'Percentage Over 200%'}, inplace=True)

# 8. Pivot the table so Year and Month are the rows, and Provider Types are columns
pivot_table = pct_over_200.pivot(
    index=['Year', 'Month_Num', 'Month'], 
    columns='Provider Type', 
    values='Percentage Over 200%'
).fillna(0).reset_index()

# 9. Sort the table chronologically by Year, then by Month Number
pivot_table = pivot_table.sort_values(['Year', 'Month_Num']).drop(columns=['Month_Num'])

# 10. Print the final result cleanly (avoiding the tabulate error!)
print(pivot_table.to_string(index=False))

 Year Month    Broker  Primary
 2017   Jan  0.000000 1.177698
 2017   Feb  0.000000 1.240777
 2017   Mar  0.000000 1.811733
 2017   Apr  0.000000 1.921585
 2017   May  0.000000 2.195311
 2017   Jun  0.000000 1.772872
 2017   Jul  0.000000 1.093621
 2017   Aug  3.000589 1.030275
 2017   Sep  3.566653 1.437484
 2017   Oct  3.089378 1.359390
 2017   Nov  3.070256 1.403692
 2017   Dec  3.612974 1.371239
 2018   Jan  3.650886 1.103027
 2018   Feb  3.217998 1.053330
 2018   Mar  3.248310 1.369734
 2018   Apr  3.443661 1.217941
 2018   May  3.705305 1.706759
 2018   Jun  2.821573 1.140290
 2018   Jul  2.464507 0.650045
 2018   Aug  2.035183 0.587267
 2018   Sep  3.013165 0.828894
 2018   Oct  3.356673 0.968458
 2018   Nov  2.774595 1.303953
 2018   Dec  2.401955 0.875750
 2019   Jan  1.973365 0.557345
 2019   Feb  2.295171 0.607413
 2019   Mar  6.843284 0.749246
 2019   Apr  5.255920 0.756964
 2019   May  5.295400 1.085992
 2019   Jun  4.940414 1.010115
 2019   Jul  4.320420 0.764912
 2019   